<a href="https://colab.research.google.com/github/fatemeh-ebadi/DataManagement/blob/main/Graph-neo-timer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
! pip install neo4j faker

In [10]:

import random
from faker import Faker
from neo4j import GraphDatabase
from datetime import timedelta
import time
import pandas as pd


fake = Faker()
fake_it = Faker('it_IT')

# 1. Connection to Neo4j Aura
NEO4J_URI = "neo4j+s://61ca1db0.databases.neo4j.io:7687"
NEO4J_USER = "61ca1db0"
NEO4J_PASSWORD = "Ad5HNN6K4O36vQhb1XJbYeBBiX9yLVwObqZUyPlfOD0"
if 'driver' in globals() and globals()['driver'] is not None:
    try:
        globals()['driver'].close()
    except:
        pass

# 2. Inputs
user_counts = [100, 500, 1000]
trip_counts = [100, 500, 1000]
events_options = [0, 2, 5, 10]
num_stations = 50

#3. Clear (Function)
def clear_graph(tx):
    # print("در حال پاکسازی دیتابیس گراف...")
    tx.run("MATCH (n) DETACH DELETE n")


# 4. Random station
def insert_stations(tx, n_stations):
    print(f"is creating {n_stations} random station...")
    for i in range(1, n_stations + 1):
        city = fake.city()
        station_name = f"station_{city}_{i}"
        lat = float(fake.latitude())
        lon = float(fake.longitude())

        tx.run("""
            MERGE (s:Station {station_id: $id})
            ON CREATE SET s.name_station = $name, s.city_station = $city,
            s.maximum_capacity = $cap, s.latitude = $lat, s.longitude = $lon
        """, id=i, name=station_name, city=city, cap=random.randint(5, 1000), lat=lat, lon=lon)


#5.Main function of simulation and combined routing (user, trip, event) - Optimized with UNWIND
def execute_simulation_round(tx, n_users, n_trips, n_events_per_trip, num_stations):
    # a. Users
    user_ids = []
    users_batch = []
    for i in range(1, n_users + 1):
        u_id = f"U{i}"
        user_ids.append(u_id)
        users_batch.append({
            "u_id": u_id,
            "name": fake.first_name(),
            "surname": fake.last_name(),
            "bdate": str(fake.date_of_birth(minimum_age=18, maximum_age=70)),
            "country": "Italy"
        })

    tx.run("""
        UNWIND $batch AS usr
        CREATE (:User {user_id: usr.u_id, name_user: usr.name, surname_user: usr.surname,
                       birthdate: usr.bdate, country_user_origin: usr.country})
    """, batch=users_batch)

    # b & c. insert trips and connection to stations and users
    trips_batch = []
    events_batch = []
    event_types = ['Delays', 'Battery', 'GPS', 'Errors']

    for j in range(1, n_trips + 1):
        t_id = f"T{j}"
        selected_user = random.choice(user_ids)
        start_station = random.randint(1, num_stations)
        end_station = random.randint(1, num_stations)

        start_time = fake.date_time_between(start_date='-30d', end_date='now')
        end_time = start_time + timedelta(hours=random.randint(1, 5))
        cost = round(random.uniform(5.0, 50.0), 2)

        # d.Create a trip to list
        trips_batch.append({
            "t_id": t_id,
            "u_id": selected_user,
            "s_start": int(start_station),
            "s_end": int(end_station),
            "s_time": start_time.isoformat(),
            "e_time": end_time.isoformat(),
            "cost": cost
        })

        # e.insert events to list in graph
        for e_idx in range(n_events_per_trip):
            e_id = f"E_{t_id}_{e_idx+1}"
            e_time = fake.date_time_between(start_date=start_time, end_date=end_time)

            events_batch.append({
                "t_id": t_id,
                "e_id": e_id,
                "e_time": e_time.isoformat(),
                "e_type": random.choice(event_types)
            })

    # for trips--> make physical graphic relationships
    tx.run("""
        UNWIND $batch AS trip
        MATCH (u:User {user_id: trip.u_id})
        MATCH (sStart:Station {station_id: toInteger(trip.s_start)})
        MATCH (sEnd:Station {station_id: toInteger(trip.s_end)})
        CREATE (t:Trip {trip_id: trip.t_id, start_time: trip.s_time, end_time: trip.e_time, total_cost: trip.cost})
        CREATE (u)-[:PERFORMED]->(t)
        CREATE (t)-[:STARTS_AT]->(sStart)
        CREATE (t)-[:ENDS_AT]->(sEnd)
    """, batch=trips_batch)

    # Events------> and connection them to trip
    if events_batch:
        tx.run("""
            UNWIND $batch AS ev
            MATCH (t:Trip {trip_id: ev.t_id})
            CREATE (e:Event {event_id: ev.e_id, timestamp_event: ev.e_time, event_type: ev.e_type})
            CREATE (t)-[:TRIGGERED]->(e)
        """, batch=events_batch)

# Function for Query1: Reachable stations for a given user
def run_query1(tx, user_id):
        query = """
        PROFILE MATCH (u:User {user_id: $u_id})-[:PERFORMED]->(t:Trip)-[:ENDS_AT]->(s:Station)
        RETURN DISTINCT s.name_station AS ReachableStations, s.city_station AS City
        """
        result = tx.run(query, u_id=user_id)
        # # To process and read the query in the database, we consume the data
        # result.consume()

        # Extract the net database processing time (in milliseconds, divided by 1000 to get seconds)
        summary = result.consume()
        # Get execution time from Neo4j metrics or fall back to result available time
        db_time_ms = summary.result_available_after
        return round(db_time_ms / 1000.0, 4)

# Function for Query2: Find 3-most important stations
def run_query2(tx):
    query = """
    MATCH (s:Station)
    OPTIONAL MATCH (s)<-[:STARTS_AT]-(t1:Trip)
    OPTIONAL MATCH (s)<-[:ENDS_AT]-(t2:Trip)
    WITH s, count(DISTINCT t1) + count(DISTINCT t2) AS total_trips
    RETURN s.name_station AS Station, total_trips
    ORDER BY total_trips DESC
    LIMIT 3
    """
    result = tx.run(query)
    summary = result.consume()

    # Process to second
    db_time_ms = summary.result_available_after
    return round(db_time_ms / 1000.0, 4)


# Main Loop
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

round_num = 1
benchmark_results = []     # A list to save benchmark results for Excel output
with driver.session() as session:
    #The values for testing
    for n_users in user_counts:
        for n_trips in trip_counts:
            for events in events_options:
                print(f"\nExecution Round #:{round_num}")
                print("------------------------------------------------------------")

                #Clear
                session.execute_write(clear_graph)
                session.execute_write(insert_stations, num_stations)

                #Start Timer
                start_time = time.time()

                session.execute_write(execute_simulation_round, n_users, n_trips, events, num_stations)
                #End Timer
                end_time = time.time()
                execution_duration = round(end_time - start_time, 4)
                print(f"Round #{round_num} finished in {execution_duration} seconds.")

                query1_duration = session.execute_read(run_query1, "U1")
                query2_duration = session.execute_read(run_query2) # <---Query2
                print(f"Q1: {query1_duration}s | Q2: {query2_duration}s")

                    # Save in Excel (Corrected with dictionary curly braces and commas)
                benchmark_results.append({
                    "Round": round_num,
                    "Num_Users": n_users,
                    "Num_Trips": n_trips,
                    "Events_Per_Trip": events,
                    "Execution_Time_Sec": execution_duration,
                    "Query1_Sec": query1_duration,
                    "Query2_Sec": query2_duration
                })
                round_num += 1

driver.close()
# تبدیل نتایج به دیتافریم و خروجی اکسل (CSV)
df = pd.DataFrame(benchmark_results)
df.to_csv("neo4j_benchmark_results.csv", index=False)
print("successfully finished!")


Execution Round #:1
------------------------------------------------------------
is creating 50 random station...
Round #1 finished in 0.4619 seconds.
Q1: 0.04s | Q2: 0.037s

Execution Round #:2
------------------------------------------------------------
is creating 50 random station...
Round #2 finished in 0.6599 seconds.
Q1: 0.0s | Q2: 0.001s

Execution Round #:3
------------------------------------------------------------
is creating 50 random station...
Round #3 finished in 0.5964 seconds.
Q1: 0.002s | Q2: 0.001s

Execution Round #:4
------------------------------------------------------------
is creating 50 random station...
Round #4 finished in 1.1875 seconds.
Q1: 0.002s | Q2: 0.001s

Execution Round #:5
------------------------------------------------------------
is creating 50 random station...
Round #5 finished in 0.5432 seconds.
Q1: 0.033s | Q2: 0.028s

Execution Round #:6
------------------------------------------------------------
is creating 50 random station...
Round #6